# HO3D Anchor Pipeline — `run_ho3d_anchor.py`

Runs and visualizes the anchor image processing step of Any6D on the HO3D dataset.

**What `run_ho3d_anchor.py` does:**
For each of the 7 YCB objects:
1. Load anchor image + depth + mask
2. Run `Any6D.register_any6d()` to estimate the 6D pose
3. Save the anchor pose (used as reference in `run_ho3d_query.py`)
4. Compute Chamfer Distance against the GT mesh
5. Save results to Excel

**Prerequisites:**
- `source ~/open-vocabulary-6d-pose-yoloe/master_env/bin/activate`
- Docker running with the `any6d` container
- Anchor results downloaded from HuggingFace
- YCB Video Models downloaded
- Kernel = `master_env`

## Cell 1 — Config & Imports

In [ ]:
import os
import subprocess
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import sys

BASE_DIR   = os.path.expanduser('~/open-vocabulary-6d-pose-yoloe')
ANY6D_DIR  = os.path.join(BASE_DIR, 'Any6D')
ANCHOR_DIR = os.path.join(ANY6D_DIR, 'anchor_results', 'dexycb_reference_view_ours')
YCB_DIR    = '/home/josue_aims_ac_za/ssd_4tb/dataset/ho3d'
HO3D_DIR   = '/home/josue_aims_ac_za/ssd_4tb/dataset/ho3d'
VIZ_DIR    = os.path.join(BASE_DIR, 'notebooks', 'outputs')
os.makedirs(VIZ_DIR, exist_ok=True)

OBJECTS = [
    '006_mustard_bottle', '021_bleach_cleanser', '019_pitcher_base',
    '004_sugar_box', '005_tomato_soup_can', '003_cracker_box', '010_potted_meat_can',
]

sys.path.insert(0, os.path.join(BASE_DIR, 'utils'))
from any6d_utils import (
    docker_run, check_docker, check_docker_volumes,
    draw_pose_axes,
    mask_overlay, colormap_depth,
    axes_legend, save_fig
)

print(f'ANY6D_DIR  : {ANY6D_DIR}')
print(f'ANCHOR_DIR : {ANCHOR_DIR}')
print(f'VIZ_DIR    : {VIZ_DIR}')

## Cell 2 — Check All Prerequisites

In [ ]:
all_ok = True
print('=== CHECKING PREREQUISITES ===')

print('\n[1] Anchor results folder:')
if os.path.exists(ANCHOR_DIR):
    obj_found = [o for o in OBJECTS if os.path.exists(os.path.join(ANCHOR_DIR, o))]
    print(f'  Found {len(obj_found)}/7 objects')
    for obj in OBJECTS:
        status = 'OK' if os.path.exists(os.path.join(ANCHOR_DIR, obj)) else 'MISSING'
        print(f'     [{status}] {obj}')
else:
    print(f'  Anchor folder not found: {ANCHOR_DIR}')
    all_ok = False

print('\n[2] YCB Video Models:')
ycb_models_path = os.path.join(YCB_DIR, 'models')
if os.path.exists(ycb_models_path):
    print(f'  Found: {len(os.listdir(ycb_models_path))} objects')
else:
    print(f'  Not found: {ycb_models_path}')
    all_ok = False

print('\n[3] HO3D Evaluation dataset:')
ho3d_eval_path = os.path.join(HO3D_DIR, 'evaluation')
if os.path.exists(ho3d_eval_path):
    print(f'  Found: {len(os.listdir(ho3d_eval_path))} sequences')
else:
    print('  Not found — needed only for run_ho3d_query.py')

print('\n[4] Docker + Any6D:')
docker_ok = check_docker(ANY6D_DIR)
if not docker_ok:
    all_ok = False

print('\n[5] Docker volume mounts:')
mounts = check_docker_volumes(ANY6D_DIR)
if not mounts.get('dataset'):
    all_ok = False

print(f'\n{"="*40}')
print('All prerequisites OK' if all_ok else 'Some prerequisites missing')

## Cell 3 — Visualize Anchor Input Data

Show the anchor image, depth map, and segmentation mask for each object before running Any6D.

In [ ]:
if not os.path.exists(ANCHOR_DIR):
    print('Anchor folder not found')
else:
    fig, axes = plt.subplots(len(OBJECTS), 3, figsize=(15, len(OBJECTS) * 3.5))

    for i, obj in enumerate(OBJECTS):
        obj_path   = os.path.join(ANCHOR_DIR, obj)
        color_path = os.path.join(obj_path, 'color.png')
        depth_path = os.path.join(obj_path, 'depth.png')
        mask_path  = os.path.join(obj_path, 'mask.png')

        if not os.path.exists(obj_path):
            for j in range(3):
                axes[i, j].text(0.5, 0.5, f'{obj}\nnot found',
                                ha='center', va='center', transform=axes[i, j].transAxes)
                axes[i, j].axis('off')
            continue

        color = cv2.cvtColor(cv2.imread(color_path), cv2.COLOR_BGR2RGB) if os.path.exists(color_path) else None

        if color is not None:
            axes[i, 0].imshow(color)
        axes[i, 0].set_title(f'{obj}\nAnchor Image', fontsize=8)
        axes[i, 0].axis('off')

        if os.path.exists(depth_path):
            depth = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH).astype(np.float32) / 1000.0
            axes[i, 1].imshow(colormap_depth(depth))
            axes[i, 1].set_title(f'Depth Map\n[{depth[depth>0].min():.2f}—{depth.max():.2f}] m', fontsize=8)
        axes[i, 1].axis('off')

        if os.path.exists(mask_path) and color is not None:
            mask      = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask_bool = mask > 127
            axes[i, 2].imshow(mask_overlay(color, mask_bool))
            axes[i, 2].set_title(f'Mask Overlay ({mask_bool.sum()} px)', fontsize=8)
        axes[i, 2].axis('off')

    plt.suptitle('Anchor Input Data — All 7 Objects')
    plt.tight_layout()
    save_fig(fig, os.path.join(VIZ_DIR, 'anchor_input_data.png'))
    plt.show()

## Cell 4 — Run `run_ho3d_anchor.py`

**Expected time: ~5-8 minutes for 7 objects**

In [ ]:
if not os.path.exists(ANCHOR_DIR):
    print('Anchor folder missing')
elif not os.path.exists(os.path.join(YCB_DIR, 'models')):
    print('YCB models missing')
else:
    print('Running run_ho3d_anchor.py in Docker...')

    r = docker_run(
        ANY6D_DIR,
        'cd /workspace && python run_ho3d_anchor.py '
        '--anchor_folder /workspace/anchor_results/dexycb_reference_view_ours '
        '--ycb_model_path /workspace/dataset/ho3d',
        timeout=600
    )

    subprocess.run(['chmod', '-R', '777', ANCHOR_DIR], capture_output=True)

    for line in r.stdout.split('\n'):
        if any(k.lower() in line.lower() for k in ['scale', 'chamfer', 'object', 'error', 'traceback', 'saved', '100%']):
            print(f'  {line.strip()}')

    if r.returncode == 0:
        print('run_ho3d_anchor.py completed successfully')
    else:
        print('Error:')
        print(r.stderr[-1000:])

## Cell 5 — Load & Display Results

In [ ]:
excel_path = os.path.join(ANCHOR_DIR, 'chamfer_distances.xlsx')

if not os.path.exists(excel_path):
    print(f'Results file not found: {excel_path}')
else:
    df = pd.read_excel(excel_path)
    print('=== CHAMFER DISTANCE RESULTS ===')
    print(f'\n{df.to_string(index=False)}')
    print(f'\nMean   : {df["Chamfer_Distance"].mean():.4f}')
    print(f'Std    : {df["Chamfer_Distance"].std():.4f}')
    print(f'Min    : {df["Chamfer_Distance"].min():.4f}')
    print(f'Max    : {df["Chamfer_Distance"].max():.4f}')

## Cell 6 — Plot Chamfer Distance Results

In [ ]:
if not os.path.exists(excel_path):
    print('Results file not found — run Cell 4 first')
else:
    df_sorted = df.sort_values('Chamfer_Distance')
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    mean_cd = df['Chamfer_Distance'].mean()
    colors  = ['#4CAF50' if v < mean_cd else '#FF5722' for v in df_sorted['Chamfer_Distance']]
    bars    = axes[0].bar(range(len(df_sorted)), df_sorted['Chamfer_Distance'],
                          color=colors, alpha=0.85, width=0.6)
    axes[0].axhline(mean_cd, color='navy', linestyle='--', linewidth=2,
                    label=f'Mean = {mean_cd:.3f}')
    axes[0].set_xticks(range(len(df_sorted)))
    axes[0].set_xticklabels([o.replace('_', '\n') for o in df_sorted['Object']], fontsize=8)
    axes[0].set_ylabel('Chamfer Distance (m)', fontsize=11)
    axes[0].set_title('Chamfer Distance per Object (green = below mean)')
    axes[0].legend(fontsize=10)
    axes[0].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, df_sorted['Chamfer_Distance']):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    stats = {'Mean': df['Chamfer_Distance'].mean(), 'Std': df['Chamfer_Distance'].std(),
             'Min': df['Chamfer_Distance'].min(), 'Max': df['Chamfer_Distance'].max(),
             'Median': df['Chamfer_Distance'].median()}
    bars2 = axes[1].bar(stats.keys(), stats.values(),
                        color=['#2196F3','#9C27B0','#4CAF50','#FF5722','#FF9800'],
                        alpha=0.85, width=0.5)
    for bar, val in zip(bars2, stats.values()):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    axes[1].set_ylabel('Chamfer Distance (m)', fontsize=11)
    axes[1].set_title('Summary Statistics')
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('Any6D — Anchor Results: Chamfer Distance')
    plt.tight_layout()
    save_fig(fig, os.path.join(VIZ_DIR, 'anchor_chamfer_results.png'))
    plt.show()

## Cell 7 — Visualize Anchor Poses on Images

Project the estimated pose axes onto each anchor image.

In [ ]:
if not os.path.exists(ANCHOR_DIR):
    print('Anchor folder not found')
else:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for i, obj in enumerate(OBJECTS):
        obj_path   = os.path.join(ANCHOR_DIR, obj)
        color_path = os.path.join(obj_path, 'color.png')
        pose_path  = os.path.join(obj_path, f'{obj}_initial_pose.txt')
        K_path     = os.path.join(obj_path, 'K.txt')

        if not all(os.path.exists(p) for p in [color_path, pose_path, K_path]):
            axes[i].text(0.5, 0.5, f'{obj}\npose not yet computed',
                         ha='center', va='center', transform=axes[i].transAxes, fontsize=9)
            axes[i].axis('off')
            continue

        color = cv2.cvtColor(cv2.imread(color_path), cv2.COLOR_BGR2RGB)
        pose  = np.loadtxt(pose_path)
        K     = np.loadtxt(K_path)

        axes[i].imshow(draw_pose_axes(color, pose, K, length=0.05))
        axes[i].set_title(f'{obj.replace("_", " ")}\nZ={pose[2,3]*100:.1f}cm', fontsize=8)
        axes[i].axis('off')

    axes[-1].axis('off')
    fig.legend(handles=[
        patches.Patch(color='#DC3232', label='X axis (Red)'),
        patches.Patch(color='#32DC32', label='Y axis (Green)'),
        patches.Patch(color='#3264E6', label='Z axis (Blue)'),
    ], loc='lower right', fontsize=10, framealpha=0.8)

    plt.suptitle('Any6D — Anchor Poses on All 7 Objects  |  Axes: object orientation in camera frame')
    plt.tight_layout()
    save_fig(fig, os.path.join(VIZ_DIR, 'anchor_poses.png'))
    plt.show()

## Cell 8 — Final Summary

In [ ]:
print('=' * 55)
print('HO3D ANCHOR PIPELINE — FINAL SUMMARY')
print('=' * 55)

print('\n  Anchor poses computed:')
for obj in OBJECTS:
    pose_path = os.path.join(ANCHOR_DIR, obj, f'{obj}_initial_pose.txt')
    if os.path.exists(pose_path):
        pose = np.loadtxt(pose_path)
        print(f'  OK  {obj:<30} Z={pose[2,3]*100:.1f}cm')
    else:
        print(f'  MISSING  {obj}')

excel_path = os.path.join(ANCHOR_DIR, 'chamfer_distances.xlsx')
if os.path.exists(excel_path):
    df = pd.read_excel(excel_path)
    print(f'\n  Chamfer Distance:')
    print(f'  Mean : {df["Chamfer_Distance"].mean():.4f} m')
    print(f'  Std  : {df["Chamfer_Distance"].std():.4f} m')

print('\n  Visualization files:')
for f in ['anchor_input_data.png', 'anchor_chamfer_results.png', 'anchor_poses.png']:
    path   = os.path.join(VIZ_DIR, f)
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {f}')

ho3d_eval = os.path.join(HO3D_DIR, 'evaluation')
print(f'\n  HO3D evaluation dataset:')
if os.path.exists(ho3d_eval):
    print('  Found — ready to run 05_ho3d_query_pipeline.ipynb')
else:
    print('  Not found — download before running query pipeline')
    print('  https://drive.google.com/drive/folders/1Wk-HZDvUExyUrRn7us4WWEbHnnFHgOAX')